In [0]:
import os, sys
import pandas as pd

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, repo_root)

from src import config_loader as cl
from src import bronze as br

In [0]:
dbutils.widgets.text("resource", "ALL", "Resource (ALL or e.g. Patient)")
dbutils.widgets.text("load_dates", "2026-11-24,2026-11-25,2026-11-26", "Load dates (comma separated)")
dbutils.widgets.text("catalog", "", "Catalog override (optional)")

resource_param = dbutils.widgets.get("resource").strip()
load_dates = [d.strip() for d in dbutils.widgets.get("load_dates").split(",") if d.strip()]
catalog_override = dbutils.widgets.get("catalog").strip()

overrides = {"project": {"catalog": catalog_override}} if catalog_override else None
cfg = cl.load_config(os.path.join(repo_root, "config", "config.yaml"), overrides)

only = None if resource_param.upper() == "ALL" else [resource_param]
resources = cl.get_resources(cfg, only)
print("Dates:", load_dates)
print("Resources:", [r["name"] for r in resources])

In [0]:
results = []
for r in resources:                       # Patient -> Encounter -> Observation -> Condition
    table = cl.resource_table(cfg, "bronze", r["name"])
    for d in load_dates:
        df, manifest = br.build_bronze_df(spark, cfg, r["name"], d)
        if df is None:
            results.append({"resource": r["name"], "load_date": d, "expected": 0,
                            "bronze_rows": 0, "status": "NO_RAW_DATA", "table": table})
            continue
        expected = br.expected_record_count(manifest)
        br.write_bronze(spark, df, table, d)
        actual = br.bronze_count(spark, table, d)
        results.append({"resource": r["name"], "load_date": d, "expected": int(expected),
                        "bronze_rows": actual,
                        "status": "OK" if actual == expected else "COUNT_MISMATCH",
                        "table": table})

res_df = pd.DataFrame(results)
display(res_df)

bad = res_df[res_df["status"] != "OK"]
if len(bad):
    raise Exception(f"Bronze load problems:\n{bad.to_string()}")

In [0]:
t = cl.resource_table(cfg, "bronze", "Patient")
display(spark.sql(f"""
  SELECT resource_id, meta_version_id, meta_last_updated, extraction_timestamp,
         api_url_or_params, raw_saved_at, bronze_loaded_at, load_date, page_number,
         substring(resource_json, 1, 120) AS resource_json_preview
  FROM {t} ORDER BY load_date, page_number LIMIT 10
"""))

display(spark.sql(f"SELECT load_date, count(*) AS rows, count(DISTINCT resource_id) AS distinct_ids FROM {t} GROUP BY load_date ORDER BY load_date"))
display(spark.sql(f"DESCRIBE HISTORY {t}").select("version", "timestamp", "operation", "operationParameters"))